# 🌫️ Bishkek Smog Predictor (PM2.5) — Kaggle/Colab OK (Sklearn 1.6+ fix)

Твоя ошибка была тут:
`TypeError: got an unexpected keyword argument 'squared'`

В sklearn 1.6+ параметр `squared=` в `mean_squared_error` больше не принимается.
В этом ноутбуке RMSE считается так: `rmse = sqrt(mean_squared_error(...))`.

Запускай сверху вниз.


In [1]:
# ===== 1) Minimal deps (do NOT pin numpy/pandas/sklearn) =====
# Keep Kaggle/Colab base env stable
!pip -q install -U --no-cache-dir catboost pyarrow


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 225.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.21.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.


In [2]:
# ===== 2) Versions =====
import sys, numpy as np, pandas as pd
import sklearn, catboost
print("python:", sys.version.split()[0])
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("sklearn:", sklearn.__version__)
print("catboost:", catboost.__version__)


python: 3.12.12
numpy: 2.0.2
pandas: 2.3.3
sklearn: 1.6.1
catboost: 1.2.10


In [3]:
# ===== 3) Write project package =====
import textwrap, sys
from pathlib import Path

ROOT = Path("./bishkek_smog_predictor")
(ROOT / "smog").mkdir(parents=True, exist_ok=True)
(ROOT / "data/processed").mkdir(parents=True, exist_ok=True)
(ROOT / "artifacts").mkdir(parents=True, exist_ok=True)

def write(rel, content):
    p = ROOT / rel
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content, encoding="utf-8")

write("smog/__init__.py", "")

write("smog/config.py", textwrap.dedent('''from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class Location:
    name: str = "Bishkek"
    latitude: float = 42.87
    longitude: float = 74.59
    timezone: str = "Asia/Bishkek"

@dataclass(frozen=True)
class Paths:
    root: Path = Path(__file__).resolve().parents[1]
    data_processed: Path = root / "data" / "processed"
    artifacts: Path = root / "artifacts"

LOC = Location()
PATHS = Paths()

ARCHIVE_WEATHER_URL = "https://archive-api.open-meteo.com/v1/archive"
AIR_QUALITY_URL = "https://air-quality-api.open-meteo.com/v1/air-quality"
FORECAST_WEATHER_URL = "https://api.open-meteo.com/v1/forecast"
'''))

write("smog/open_meteo.py", textwrap.dedent('''from __future__ import annotations
import time
from typing import Any, Dict, Optional
import requests

class OpenMeteoError(RuntimeError):
    pass

def get_json(url: str, params: Dict[str, Any], timeout: int = 30, retries: int = 3, backoff: float = 1.5) -> Dict[str, Any]:
    last_err: Optional[Exception] = None
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(url, params=params, timeout=timeout)
            if r.status_code != 200:
                raise OpenMeteoError(f"HTTP {r.status_code}: {r.text[:300]}")
            data = r.json()
            if isinstance(data, dict) and data.get("error"):
                raise OpenMeteoError(f"API error: {data.get('reason')}")
            return data
        except Exception as e:
            last_err = e
            if attempt < retries:
                time.sleep(backoff ** attempt)
            else:
                raise OpenMeteoError(f"Failed GET {url} after {retries} retries. Last error: {e}") from e
    raise OpenMeteoError(str(last_err))
'''))

write("smog/ingest.py", textwrap.dedent('''from __future__ import annotations
from datetime import date
import pandas as pd
from .config import LOC, PATHS, ARCHIVE_WEATHER_URL, AIR_QUALITY_URL
from .open_meteo import get_json

def fetch_hourly_weather(start: str, end: str) -> pd.DataFrame:
    params = {
        "latitude": LOC.latitude,
        "longitude": LOC.longitude,
        "start_date": start,
        "end_date": end,
        "timezone": LOC.timezone,
        "hourly": ",".join([
            "temperature_2m",
            "relative_humidity_2m",
            "precipitation",
            "rain",
            "snowfall",
            "surface_pressure",
            "wind_speed_10m",
            "wind_direction_10m",
            "wind_gusts_10m",
        ]),
    }
    js = get_json(ARCHIVE_WEATHER_URL, params=params)
    df = pd.DataFrame(js["hourly"])
    df["time"] = pd.to_datetime(df["time"])
    return df

def fetch_hourly_pm25(start: str, end: str) -> pd.DataFrame:
    params = {
        "latitude": LOC.latitude,
        "longitude": LOC.longitude,
        "start_date": start,
        "end_date": end,
        "timezone": LOC.timezone,
        "hourly": "pm2_5",
    }
    js = get_json(AIR_QUALITY_URL, params=params)
    df = pd.DataFrame(js["hourly"])
    df["time"] = pd.to_datetime(df["time"])
    return df

def hourly_to_daily_weather(df_h: pd.DataFrame) -> pd.DataFrame:
    df = df_h.copy()
    df["date"] = df["time"].dt.date
    is_night = (df["time"].dt.hour <= 7) | (df["time"].dt.hour >= 20)

    def agg_block(x: pd.DataFrame, suffix: str) -> pd.DataFrame:
        g = x.groupby("date").agg(
            temp_mean=("temperature_2m", "mean"),
            temp_min=("temperature_2m", "min"),
            temp_max=("temperature_2m", "max"),
            rh_mean=("relative_humidity_2m", "mean"),
            rh_max=("relative_humidity_2m", "max"),
            pressure_mean=("surface_pressure", "mean"),
            wind_mean=("wind_speed_10m", "mean"),
            wind_max=("wind_speed_10m", "max"),
            gust_max=("wind_gusts_10m", "max"),
            wind_dir_mean=("wind_direction_10m", "mean"),
            precip_sum=("precipitation", "sum"),
            rain_sum=("rain", "sum"),
            snowfall_sum=("snowfall", "sum"),
        ).reset_index()
        g = g.rename(columns={c: f"{c}{suffix}" for c in g.columns if c != "date"})
        return g

    day = agg_block(df[~is_night], "_day")
    night = agg_block(df[is_night], "_night")
    out = day.merge(night, on="date", how="outer").sort_values("date")
    out["temp_range_day"] = out["temp_max_day"] - out["temp_min_day"]
    out["temp_range_night"] = out["temp_max_night"] - out["temp_min_night"]
    return out

def hourly_to_daily_pm25(df_h: pd.DataFrame) -> pd.DataFrame:
    df = df_h.copy()
    df["date"] = df["time"].dt.date
    return df.groupby("date").agg(
        pm25_mean=("pm2_5", "mean"),
        pm25_max=("pm2_5", "max"),
    ).reset_index()

def build_daily_dataset(start: str, end: str) -> pd.DataFrame:
    w_d = hourly_to_daily_weather(fetch_hourly_weather(start, end))
    a_d = hourly_to_daily_pm25(fetch_hourly_pm25(start, end))
    df = w_d.merge(a_d, on="date", how="inner").sort_values("date")
    df["date"] = pd.to_datetime(df["date"])
    return df

def run_ingest(start="2022-08-01", end=None):
    if end is None:
        end = str(date.today())
    df = build_daily_dataset(start, end)
    PATHS.data_processed.mkdir(parents=True, exist_ok=True)
    out = PATHS.data_processed / "bishkek_daily.parquet"
    df.to_parquet(out, index=False)
    return out, df
'''))

write("smog/features.py", textwrap.dedent('''from __future__ import annotations
import numpy as np
import pandas as pd

def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    d = out["date"]
    out["dow"] = d.dt.dayofweek
    out["month"] = d.dt.month
    out["dayofyear"] = d.dt.dayofyear
    out["doy_sin"] = np.sin(2 * np.pi * out["dayofyear"] / 365.25)
    out["doy_cos"] = np.cos(2 * np.pi * out["dayofyear"] / 365.25)
    out["is_weekend"] = (out["dow"] >= 5).astype(int)
    out["heating_season"] = out["month"].isin([10,11,12,1,2,3]).astype(int)
    return out

def add_pm_lag_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["pm25_lag1"] = out["pm25_mean"].shift(1)
    out["pm25_lag2"] = out["pm25_mean"].shift(2)
    out["pm25_lag7"] = out["pm25_mean"].shift(7)
    out["pm25_roll3"] = out["pm25_mean"].shift(1).rolling(3).mean()
    out["pm25_roll7"] = out["pm25_mean"].shift(1).rolling(7).mean()
    out["pm25_roll14"] = out["pm25_mean"].shift(1).rolling(14).mean()
    return out

def add_wind_dir_encoding(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in ["wind_dir_mean_day", "wind_dir_mean_night"]:
        if col in out.columns:
            rad = np.deg2rad(out[col].fillna(0))
            out[col + "_sin"] = np.sin(rad)
            out[col + "_cos"] = np.cos(rad)
    return out

def make_training_frame(df_daily: pd.DataFrame) -> pd.DataFrame:
    df = df_daily.copy().sort_values("date").reset_index(drop=True)
    df = add_time_features(df)
    df = add_pm_lag_features(df)
    df = add_wind_dir_encoding(df)
    return df

def _unique(seq):
    seen = set()
    out = []
    for x in seq:
        if x not in seen:
            out.append(x)
            seen.add(x)
    return out

def get_feature_columns(df: pd.DataFrame) -> list[str]:
    weather_cols = [
        "temp_mean_day","temp_min_day","temp_max_day","rh_mean_day","rh_max_day",
        "pressure_mean_day","wind_mean_day","wind_max_day","gust_max_day",
        "precip_sum_day","rain_sum_day","snowfall_sum_day","temp_range_day",
        "temp_mean_night","temp_min_night","temp_max_night","rh_mean_night","rh_max_night",
        "pressure_mean_night","wind_mean_night","wind_max_night","gust_max_night",
        "precip_sum_night","rain_sum_night","snowfall_sum_night","temp_range_night",
    ]
    wind_dir_cols = [c for c in df.columns if c.startswith("wind_dir_mean_") and (c.endswith("_sin") or c.endswith("_cos"))]
    base = [
        "dow","month","doy_sin","doy_cos","is_weekend","heating_season",
        "pm25_lag1","pm25_lag2","pm25_lag7","pm25_roll3","pm25_roll7","pm25_roll14",
    ]
    cols = [c for c in base + weather_cols + wind_dir_cols if c in df.columns]
    return _unique(cols)
'''))

write("smog/train.py", textwrap.dedent('''from __future__ import annotations
import json
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from .config import PATHS
from .features import make_training_frame, get_feature_columns

def _metrics(y_true, y_pred) -> dict:
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    mse = mean_squared_error(y_true, y_pred)  # sklearn 1.6+ compatible
    rmse = float(np.sqrt(mse))
    return {
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": rmse,
        "r2": float(r2_score(y_true, y_pred)),
    }

def run_train(verbose=200):
    data_path = PATHS.data_processed / "bishkek_daily.parquet"
    df0 = pd.read_parquet(data_path)
    df = make_training_frame(df0).dropna().reset_index(drop=True)

    feature_cols = get_feature_columns(df)
    X = df[feature_cols]
    y = df["pm25_mean"]

    tscv = TimeSeriesSplit(n_splits=5)
    cv_scores = []
    for fold, (tr, va) in enumerate(tscv.split(X), 1):
        model = CatBoostRegressor(
            loss_function="MAE",
            iterations=3000,
            learning_rate=0.03,
            depth=7,
            l2_leaf_reg=5,
            random_seed=42,
            verbose=False,
        )
        model.fit(X.iloc[tr], y.iloc[tr], eval_set=(X.iloc[va], y.iloc[va]),
                  use_best_model=True, early_stopping_rounds=150)
        pred = model.predict(X.iloc[va])
        m = _metrics(y.iloc[va], pred)
        m["fold"] = fold
        m["best_iteration"] = int(model.get_best_iteration())
        cv_scores.append(m)

    cv_mean = {
        "mae": float(np.mean([m["mae"] for m in cv_scores])),
        "rmse": float(np.mean([m["rmse"] for m in cv_scores])),
        "r2": float(np.mean([m["r2"] for m in cv_scores])),
    }

    split = int(len(df) * 0.8)
    X_tr, X_te = X.iloc[:split], X.iloc[split:]
    y_tr, y_te = y.iloc[:split], y.iloc[split:]

    final = CatBoostRegressor(
        loss_function="MAE",
        iterations=6000,
        learning_rate=0.03,
        depth=7,
        l2_leaf_reg=5,
        random_seed=42,
        verbose=verbose,
    )
    final.fit(X_tr, y_tr, eval_set=(X_te, y_te),
              use_best_model=True, early_stopping_rounds=250)
    pred_te = final.predict(X_te)
    hold = _metrics(y_te, pred_te)
    hold["best_iteration"] = int(final.get_best_iteration())

    PATHS.artifacts.mkdir(parents=True, exist_ok=True)
    model_path = PATHS.artifacts / "bishkek_pm25_catboost.cbm"
    feats_path = PATHS.artifacts / "feature_columns.json"
    metrics_path = PATHS.artifacts / "metrics.json"

    final.save_model(model_path)
    feats_path.write_text(json.dumps(feature_cols, ensure_ascii=False, indent=2), encoding="utf-8")
    metrics_path.write_text(json.dumps({
        "cv_mean": cv_mean,
        "holdout": hold,
        "n_rows": int(len(df)),
        "n_features": int(len(feature_cols)),
    }, ensure_ascii=False, indent=2), encoding="utf-8")

    return model_path, feats_path, metrics_path, cv_mean, hold
'''))

sys.path.insert(0, str(ROOT))
print("✅ Project written to:", ROOT)


✅ Project written to: bishkek_smog_predictor


In [4]:
# ===== 4) Ingest history dataset =====
from smog.ingest import run_ingest
out_path, df_daily = run_ingest(start="2022-08-01")
print("Saved:", out_path)
df_daily.tail()


Saved: /kaggle/working/bishkek_smog_predictor/data/processed/bishkek_daily.parquet


,date,temp_mean_day,temp_min_day,temp_max_day,rh_mean_day,rh_max_day,pressure_mean_day,wind_mean_day,wind_max_day,gust_max_day,...,wind_max_night,gust_max_night,wind_dir_mean_night,precip_sum_night,rain_sum_night,snowfall_sum_night,temp_range_day,temp_range_night,pm25_mean,pm25_max
1304,2026-02-25,4.975000,1.2,8.9,85.666667,99,926.800000,4.833333,8.1,20.5,...,9.6,18.7,205.666667,1.4,1.4,0.00,7.7,1.6,17.925000,21.1
1305,2026-02-26,3.600000,1.6,4.9,89.000000,99,925.991667,7.575000,19.8,46.8,...,8.9,17.6,223.500000,1.7,1.4,0.21,3.3,1.9,6.887500,16.5
1306,2026-02-27,4.391667,2.0,7.7,74.333333,92,928.516667,6.516667,11.9,29.2,...,6.0,12.2,208.666667,0.9,0.8,0.07,5.7,1.8,11.187500,29.3
1307,2026-02-28,-3.258333,-4.4,-2.0,86.916667,97,934.300000,6.150000,15.5,33.1,...,10.2,25.2,258.750000,1.9,0.1,1.26,2.4,3.1,12.837500,29.1
1308,2026-03-01,0.025000,-4.9,3.7,76.916667,95,932.875000,3.200000,4.5,18.0,...,5.8,14.4,218.166667,0.0,0.0,0.00,8.6,5.1,11.733333,16.9


In [5]:
# ===== 5) Train model =====
from smog.train import run_train
model_path, feats_path, metrics_path, cv_mean, hold = run_train(verbose=200)
print("Model:", model_path)
print("CV mean:", cv_mean)
print("Holdout:", hold)


0:	learn: 3.8032456	test: 5.9058430	best: 5.9058430 (0)	total: 11.6ms	remaining: 1m 9s
200:	learn: 1.5075381	test: 3.1941150	best: 3.1940161 (198)	total: 1.87s	remaining: 53.9s
400:	learn: 1.0399822	test: 3.1475382	best: 3.1388717 (359)	total: 3.58s	remaining: 50s
600:	learn: 0.8042247	test: 3.1354376	best: 3.1337904 (595)	total: 5.29s	remaining: 47.5s
800:	learn: 0.6491121	test: 3.1230226	best: 3.1208415 (788)	total: 7.02s	remaining: 45.6s
1000:	learn: 0.5490803	test: 3.1207003	best: 3.1175723 (910)	total: 8.76s	remaining: 43.8s
1200:	learn: 0.4716800	test: 3.1031771	best: 3.1018100 (1162)	total: 10.5s	remaining: 41.9s
1400:	learn: 0.4181174	test: 3.0983515	best: 3.0974358 (1388)	total: 12.3s	remaining: 40.3s
1600:	learn: 0.3791613	test: 3.0934090	best: 3.0922430 (1582)	total: 14s	remaining: 38.4s
1800:	learn: 0.3422111	test: 3.0922771	best: 3.0920838 (1711)	total: 15.7s	remaining: 36.7s
2000:	learn: 0.3123887	test: 3.0871363	best: 3.0864284 (1902)	total: 17.5s	remaining: 34.9s
Stoppe